# M11.1 · Rastgele süreçler ve Markov zincirleri

Pekiştirmeli öğrenmenin olasılık zemini: rastgele değişken ve rastgele süreç, birinci ve ikinci derece dağılımlar, ortalama / otokorelasyon / otokovaryans, katı ve geniş anlamda durağanlık, Wiener süreci; ardından durum geçişlerini modelleyen birinci dereceden durağan Markov zincirleri: geçiş matrisi, n adımlık dağılım, durağan dağılım. Saf Python'da zincir ve AR(1) simülasyonu; NumPy'da matris kuvvetleri, özvektörle durağan dağılım ve topluluk istatistikleri; PyTorch'ta binlerce zinciri birden örnekleyip geçiş matrisini en çok olabilirlikle hem sayarak hem gradyanla kestiriyorsun; en sonda Wiener sürecinin özelliklerini (varyans, karesel değişim, yansıma ilkesi) vektörel simülasyonla doğruluyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m11-1-markov.html](https://m-gonul.github.io/pytorch-lab/m11-1-markov.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m11_1_markov.py
CHECKS = {
 "b": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"scipy\", \"torch\"}))\n\n    def si():\n        f = need(ns, \"sample_index\")\n        ensure([f([0.2, 0.5, 0.3], u) for u in (0.0, 0.19, 0.2, 0.69, 0.7, 0.999)] == [0, 0, 1, 1, 2, 2], \"ters CDF\")\n    R.test(\"sample_index\", si)\n\n    def ch():\n        T = [[0.7, 0.2, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]]\n        s = need(ns, \"simulate_chain\")(T, 0, 60000, 1)\n        ensure(len(s) == 60001 and s[0] == 0, \"uzunluk ve başlangıç\")\n        Tn = np.array(T)\n        w, V = np.linalg.eig(Tn); pi = np.real(V[:, np.argmin(abs(w - 1))]); pi /= pi.sum()\n        freq = np.bincount(s, minlength=3) / len(s)\n        close(freq, pi, atol=0.015, what=\"uzun dönem frekanslar ≈ durağan dağılım\")\n        a = np.array(s)\n        emp = np.array([[((a[:-1] == j) & (a[1:] == i)).sum() for j in range(3)] for i in range(3)], float)\n        emp /= emp.sum(0, keepdims=True)\n        close(emp, Tn, atol=0.02, what=\"ampirik geçiş matrisi\")\n        return f\"frekanslar {np.round(freq, 3).tolist()}, durağan {np.round(pi, 3).tolist()}\"\n    R.test(\"simulate_chain\", ch)\n\n    def ar():\n        x = need(ns, \"ar1\")(0.8, 1.0, 50000, 3)\n        xs = np.array(x[1000:])\n        close(xs.var(), 1 / 0.36, rtol=0.08, what=\"durağan varyans\")\n        for lag in (1, 2, 5):\n            close(need(ns, \"autocov\")(x, lag), np.mean((np.array(x[:-lag]) - np.mean(x)) * (np.array(x[lag:]) - np.mean(x))) * (len(x) - lag) / len(x), atol=1e-9, what=f\"autocov {lag}\")\n        r = [ns[\"autocov\"](x[1000:], k) / ns[\"autocov\"](x[1000:], 0) for k in range(4)]\n        close(np.array(r), 0.8 ** np.arange(4), atol=0.05, what=\"otokorelasyon ≈ φ^τ\")\n    R.test(\"ar1 ve autocov\", ar)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    rng = np.random.default_rng(0)\n    T = rng.random((5, 5)); T /= T.sum(0, keepdims=True)\n    p0 = np.eye(5)[2]\n\n    def ns_():\n        close(need(ns, \"n_step\")(T, p0, 7), np.linalg.matrix_power(T, 7) @ p0, what=\"T^7 p0\")\n        pi = need(ns, \"stationary\")(T)\n        close(T @ pi, pi, atol=1e-12, what=\"Tπ = π\"); close(pi.sum(), 1.0, what=\"toplam 1\")\n    R.test(\"n_step ve stationary\", ns_)\n\n    def tv():\n        c = need(ns, \"tv_curve\")(T, p0, 20)\n        want = [0.5 * np.abs(np.linalg.matrix_power(T, t) @ p0 - ns[\"stationary\"](T)).sum() for t in range(21)]\n        close(c, np.array(want), atol=1e-10, what=\"TV eğrisi\")\n        lam2 = sorted(np.abs(np.linalg.eigvals(T)))[-2]\n        R.table(\"Karışma (mixing)\", [\"t\", \"TV uzaklığı\"], [[t, f\"{c[t]:.2e}\"] for t in (0, 1, 2, 5, 10, 20)],\n                note=f\"Uzaklık kabaca |λ₂|^t = {lam2:.3f}^t hızıyla azalır: ikinci en büyük özdeğer karışma hızını belirler.\")\n    R.test(\"tv_curve (özdeğer ayrışımıyla)\", tv)\n\n    def ar():\n        X = need(ns, \"ar1_paths\")(0.8, 1.0, 200, 4000, 1)\n        ensure(X.shape == (4000, 200) and np.all(X[:, 0] == 0), \"şekil ve x0 = 0\")\n        m, v = need(ns, \"ensemble_stats\")(X)\n        theory = (1 - 0.64 ** np.arange(200)) / (1 - 0.64)\n        close(v[1:], theory[1:], rtol=0.1, what=\"Var x_t = (1 − φ^{2t})/(1 − φ²)\")\n        ensure(np.abs(m).max() < 0.12, \"topluluk ortalaması ≈ 0\")\n        return f\"t = 1: var {v[1]:.3f}, t = 199: var {v[199]:.3f} (durağan değer {1 / 0.36:.3f}) — başlangıçta durağan değil, sonra WSS\"\n    R.test(\"ar1_paths ve ensemble_stats\", ar)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    g = torch.Generator().manual_seed(0)\n    T = torch.rand(4, 4, generator=g, dtype=torch.float64); T /= T.sum(0, keepdim=True)\n\n    def sim():\n        tr = need(ns, \"simulate\")(T, 1, 200, 500, 3)\n        ensure(tuple(tr.shape) == (500, 201) and bool((tr[:, 0] == 1).all()), \"şekil\")\n        Th = need(ns, \"count_mle\")(tr, 4)\n        close(Th.sum(0).numpy(), torch.ones(4).numpy(), atol=1e-12, what=\"sütun toplamları 1\")\n        close(Th.numpy(), T.numpy(), atol=0.02, what=\"100 bin geçişten kestirim\")\n    R.test(\"simulate ve count_mle\", sim)\n\n    def gm():\n        tr = ns[\"simulate\"](T, 0, 100, 50, 5)\n        a = ns[\"count_mle\"](tr, 4)\n        b = need(ns, \"grad_mle\")(tr, 4)\n        close(b.numpy(), a.numpy(), atol=2e-3, what=\"gradyanla MLE = sayma MLE\")\n        rows = []\n        for n in (10, 100, 1000):\n            e = (ns[\"count_mle\"](ns[\"simulate\"](T, 0, 100, n, 7), 4) - T).abs().max().item()\n            rows.append([n * 100, f\"{e:.4f}\"])\n        R.table(\"Kestirim hatası (en büyük mutlak)\", [\"geçiş sayısı\", \"hata\"], rows, note=\"Hata ≈ 1/√n ile azalır: 100 kat veri, 10 kat daha küçük hata.\")\n    R.test(\"grad_mle\", gm)\n",
 "e": "import math\nimport torch\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n\n    def var():\n        W = need(ns, \"wiener\")(20000, 500, 2.0, 1)\n        ensure(tuple(W.shape) == (20000, 501) and bool((W[:, 0] == 0).all()), \"şekil, W0 = 0\")\n        for k in (100, 250, 500):\n            t = 2.0 * k / 500\n            close(W[:, k].var().item(), t, rtol=0.05, what=f\"Var W({t})\")\n        close(torch.cov(torch.stack([W[:, 100], W[:, 400]])).numpy()[0, 1], 0.4, atol=0.03, what=\"Cov(W(0.4), W(1.6)) = min(s, t)\")\n    R.test(\"Wiener: Var W(t) = t, Cov = min(s, t)\", var)\n\n    def qv():\n        rows = []\n        for n in (10, 100, 1000):\n            q = need(ns, \"quadratic_variation\")(ns[\"wiener\"](5000, n, 1.0, 2))\n            rows.append([n, f\"{q.mean().item():.4f}\", f\"{q.std().item():.4f}\"])\n        ensure(float(rows[-1][2]) < float(rows[0][2]) / 5, \"karesel değişimin varyansı adım sayısıyla sıfıra gitmeli\")\n        R.table(\"Karesel değişim Σ(ΔW)², t = 1\", [\"adım\", \"ortalama\", \"std\"], rows, note=\"Pürüzsüz fonksiyonlarda karesel değişim 0'dır; Wiener sürecinde deterministik olarak t'dir (Itô hesabının temeli).\")\n    R.test(\"quadratic_variation\", qv)\n\n    def refl():\n        W = ns[\"wiener\"](20000, 2000, 1.0, 3)\n        a, b = need(ns, \"max_exceed\")(W, 1.0)\n        close(b, 2 * (1 - 0.5 * (1 + math.erf(1 / math.sqrt(2)))), atol=0.015, what=\"2P(W ≥ 1)\")\n        ensure(abs(a - b) < 0.03, f\"yansıma ilkesi: P(max ≥ 1) = {a:.3f}, 2P(W ≥ 1) = {b:.3f}\")\n    R.test(\"yansıma ilkesi\", refl)\n    if \"wiener\" not in ns:\n        return\n\n    def loop():\n        g = torch.Generator().manual_seed(0)\n        out = torch.zeros(2000, 501, dtype=torch.float64)\n        for t in range(500):\n            out[:, t + 1] = out[:, t] + torch.randn(2000, generator=g, dtype=torch.float64) * (1 / 500) ** 0.5\n        return out\n    R.table(\"2000 yol × 500 adım\", [\"yöntem\", \"süre\"], [[\"adım döngüsü\", fmt_time(bench(loop, repeat=1, number=1))], [\"cumsum\", fmt_time(bench(lambda: ns[\"wiener\"](2000, 500), repeat=3))]])\n",
 "x1": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    def gr():\n        rows = []\n        for N, p in [(10, 0.5), (10, 0.45), (20, 0.55)]:\n            got = need(ns, \"gamblers_ruin\")(N, p)\n            i = np.arange(1, N)\n            q = 1 - p\n            want = i / N if p == 0.5 else (1 - (q / p) ** i) / (1 - (q / p) ** N)\n            close(got, want, atol=1e-10, what=f\"N = {N}, p = {p}\")\n            rows.append([N, p, f\"{got[N // 2 - 1]:.4f}\"])\n        R.table(\"Kumarbazın iflası: yarı servetle başlayıp hedefe ulaşma olasılığı\", [\"N\", \"p\", \"P(hedef)\"], rows, note=\"p = 0.45'te bile olasılık hızla düşer: küçük bir dezavantaj uzun oyunda belirleyici.\")\n    R.test(\"gamblers_ruin (soğurulma olasılıkları)\", gr)\n\n    def ab():\n        P = np.zeros((7, 7)); P[0, 0] = P[6, 6] = 1\n        for i in range(1, 6):\n            P[i, i - 1] = P[i, i + 1] = 0.5\n        B, t = need(ns, \"absorption\")(P, [0, 6])\n        close(t, np.array([i * (6 - i) for i in range(1, 6)], float), atol=1e-10, what=\"beklenen adım i(N − i)\")\n    R.test(\"absorption (beklenen adım)\", ab)\n\n    def mf():\n        rng = np.random.default_rng(0)\n        P = rng.random((5, 5)); P /= P.sum(1, keepdims=True)\n        M = need(ns, \"mean_first_passage\")(P)\n        w, V = np.linalg.eig(P.T); pi = np.real(V[:, np.argmin(abs(w - 1))]); pi /= pi.sum()\n        close(np.diag(M), 1 / pi, atol=1e-9, what=\"Kac: m_jj = 1/π_j\")\n        s, total, cnt = 0, 0, 0\n        steps = 0\n        for _ in range(20000):\n            nxt = rng.choice(5, p=P[s]); steps += 1\n            if nxt == 3:\n                total += steps; cnt += 1; steps = 0\n            s = nxt\n        close(total / cnt, M[3, 3] if True else 0, rtol=0.05, what=\"simülasyonla dönüş zamanı\")\n    R.test(\"mean_first_passage\", mf)\n",
 "x2": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    w = 0.3\n\n    def t():\n        X = need(ns, \"random_phase\")(20000, 60, w, 1)\n        Y = need(ns, \"random_amplitude\")(20000, 60, w, 2)\n        Z = need(ns, \"random_constant\")(2000, 500, 3)\n        vX, vY = X.var(0), Y.var(0)\n        ensure(vX.std() < 0.02, \"rastgele fazlı sürecin varyansı zamanla sabit (WSS)\")\n        close(vY, np.cos(w * np.arange(60)) ** 2, atol=0.05, what=\"rastgele genlik: Var = cos²(ωt)\")\n        cX = np.mean(X[:, 10] * X[:, 15]); cX2 = np.mean(X[:, 30] * X[:, 35])\n        close(cX, 0.5 * np.cos(w * 5), atol=0.02, what=\"kovaryans yalnız τ'ya bağlı\"); close(cX2, cX, atol=0.03, what=\"kayma değişmezliği\")\n        tz, ez = need(ns, \"time_vs_ensemble\")(Z)\n        tx, ex = ns[\"time_vs_ensemble\"](X[:, :60])\n        ensure(tz > 0.9 and ez < 0.1, \"sabit süreç: zaman ortalamaları yola göre çok değişir (ergodik değil)\")\n        R.table(\"Süreçler\", [\"süreç\", \"WSS?\", \"ergodik (ortalamada)?\", \"zaman ort. std\", \"topluluk ort. std\"],\n                [[\"cos(ωt + θ)\", \"evet\", \"evet\", f\"{tx:.3f}\", f\"{ex:.3f}\"], [\"A·cos(ωt)\", \"hayır\", \"—\", \"—\", \"—\"], [\"x_t = A\", \"evet (katı)\", \"hayır\", f\"{tz:.3f}\", f\"{ez:.3f}\"]])\n    R.test(\"durağanlık ve ergodiklik\", t)\n",
 "x3": "import math\nfrom labcheck import close, need, ensure\n\nTEXT = (\n    \"pekiştirmeli öğrenmede bir ajan bir ortamla etkileşir. ajan her adımda ortamın durumunu gözler, \"\n    \"bir eylem seçer ve bir ödül alır. amaç uzun vadede toplanan ödülü en büyük yapmaktır. ortamın \"\n    \"bir sonraki durumu yalnızca şimdiki duruma ve seçilen eyleme bağlıysa ortam markov özelliğini \"\n    \"taşır. bu özellik sayesinde geçmişin tamamını hatırlamak gerekmez; şimdiki durum geleceği \"\n    \"tahmin etmek için yeterlidir. değer fonksiyonu bir durumdan başlayarak beklenen toplam ödülü \"\n    \"ölçer. politika ise her durumda hangi eylemin seçileceğini söyler. iyi bir politika bulmak için \"\n    \"ajan hem bildiklerini kullanmalı hem de yeni şeyler denemelidir. bu ikilem keşif ve sömürü \"\n    \"dengesi olarak bilinir ve pekiştirmeli öğrenmenin merkezindedir. \"\n) * 3\n\n\ndef check(ns, R):\n    train, test = TEXT[:1400], TEXT[1400:1700]\n    vocab = sorted(set(TEXT))\n\n    def f():\n        m = need(ns, \"fit\")(\"abracadabra\", 1)\n        ensure(dict(m[\"a\"]) == {\"b\": 2, \"c\": 1, \"d\": 1}, f\"'a' bağlamı: {dict(m['a'])}\")\n        m2 = ns[\"fit\"](\"abracadabra\", 2)\n        ensure(dict(m2[\"ab\"]) == {\"r\": 2}, \"2. derece\")\n    R.test(\"fit\", f)\n\n    def lp():\n        m = ns[\"fit\"](train, 1)\n        V = len(vocab)\n        c = m.get(\"a\", {}); want = math.log((c.get(\"b\", 0) + 1) / (sum(c.values()) + V))\n        close(need(ns, \"log_prob\")(m, \"ab\", 1, 1.0, vocab), want, atol=1e-12, what=\"tek geçiş\")\n        rows = []\n        for k in (0, 1, 2, 3, 5):\n            mk = ns[\"fit\"](train, k)\n            bpc = -ns[\"log_prob\"](mk, test, k, 0.1, vocab) / (len(test) - k) / math.log(2)\n            rows.append([k, f\"{bpc:.2f}\"])\n        ensure(float(rows[2][1]) < float(rows[0][1]), \"bağlam test kayıbını azaltmalı\")\n        R.table(\"Test karakter başına bit (α = 0.1)\", [\"derece k\", \"bit / karakter\"], rows,\n                note=\"k büyüdükçe önce iyileşir; çok büyük k'da bağlamlar seyrekleşir (bu küçük metin tekrarlı olduğu için ezber de mümkün). Dil modellerinin (M4) çıkış noktası.\")\n    R.test(\"log_prob\", lp)\n\n    def g():\n        m = ns[\"fit\"](train, 3)\n        s = need(ns, \"generate\")(m, \"pek\", 80, 3, 1)\n        ensure(len(s) == 83 and s.startswith(\"pek\"), \"uzunluk\")\n        ensure(all(s[i:i + 3] in m or True for i in range(len(s) - 3)), \"\")\n        return f\"örnek: «{s}»\"\n    R.test(\"generate\", g)\n"
}

In [ ]:
import importlib, labcheck, checks_m11_1_markov as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Rastgele değişken:** bir deneyin her sonucuna $\zeta$ bir sayı atayan kural, $\mathbf x(\zeta)$. **Rastgele süreç:** her sonuca bir **zaman fonksiyonu** atayan kural $\mathbf x(t,\zeta)$. Sabit $\zeta$ → bir örnek yol (gerçekleşme); sabit $t$ → bir rastgele değişken. Borsa fiyatları, günlük sıcaklık, gürültülü ölçümler…

**İstatistikler:** birinci derece dağılım $F(x;t)=P\{\mathbf x(t)\le x\}$, ikinci derece $F(x_1,x_2;t_1,t_2)$; ortalama $\eta(t)=E\{\mathbf x(t)\}$, otokorelasyon $R(t_1,t_2)=E\{\mathbf x(t_1)\mathbf x(t_2)\}$, otokovaryans $C(t_1,t_2)=R(t_1,t_2)-\eta(t_1)\eta(t_2)$.

**Durağanlık:** *katı* — tüm sonlu boyutlu dağılımlar zaman kaymasına değişmez. *Geniş anlamda (WSS)* — ortalama sabit, otokovaryans yalnız $\tau=t_1-t_2$'ye bağlı, ikinci moment sonlu. Gerçek sistemler nadiren katı durağandır; WSS çoğu zaman yeter. **Ergodiklik:** zaman ortalaması topluluk ortalamasına eşit (tek uzun gerçekleşmeden istatistik kestirilebilir).

**Wiener süreci** $W(t)$: $W(0)=0$, bağımsız artışlar, $W(t)-W(s)\sim\mathcal N(0,t-s)$, sürekli yollar. Rastgele yürüyüşün sürekli limiti.

**RL bağlantısı:** gözle → eylem seç → ödülü gör → tekrarla döngüsü, belirsizlikler yüzünden bir rastgele süreçtir. İlk model yalnız durum geçişleri: $s_0\to s_1\to s_2\to\cdots$.

**Markov süreçleri:** geçmişin tamamına bağlı $P(s_t|s_{t-1},…,s_0)$ modellemek imkansız; **k. derece Markov**: yalnız son $k$ durum; **birinci derece**: $P(s_t|s_{t-1})$ ("gelecek, şimdi verildiğinde geçmişten bağımsız"). **Durağan (zamanla homojen)**: geçiş olasılıkları $t$'ye bağlı değil. Durum kümesi sonlu ya da sayılabilir ise **Markov zinciri** — geçişler matris–vektör çarpımı:

$$T_{ij}=P(s_{t+1}=S_i\mid s_t=S_j),\qquad\mathbf p_{t+1}=T\mathbf p_t,\qquad\mathbf p_n=T^n\mathbf p_0.$$

(Bu modülde sütunlar "nereden", satırlar "nereye"; sütun toplamları 1. MDP modüllerinde satır-stokastik $P=T^\top$ da kullanılacak.) **Durağan dağılım:** $T\boldsymbol\pi=\boldsymbol\pi$ — özdeğer 1'in özvektörü; indirgenemez ve periyodik olmayan zincirde tek ve her başlangıçtan yakınsanır.

:::sezgi Pekiştirmeli öğrenme nerede duruyor?
**Denetimli öğrenme:** her girdi için doğru cevap (etiket) verilir; hata doğrudan ölçülür. **Denetimsiz:** etiket yok; verinin yapısı (küme, yoğunluk, düşük boyutlu temsil) aranır. **Yarı denetimli:** az etiketli + çok etiketsiz veri. **Pekiştirmeli öğrenme:** doğru cevap yok, yalnız bir **ödül sinyali** var — üstelik gecikmeli (iyi bir hamlenin değeri oyunun sonunda belli olur) ve ajanın kendi eylemleri hangi veriyi göreceğini belirler (veri i.i.d. değil). Kökü hayvan psikolojisinde: Thorndike'ın etki yasası, Pavlov ve Skinner'ın koşullanması, ödül tahmin hatasının dopamin nöronlarıyla ilişkisi (TD hatası, M11.4). Uygulamalar: oyunlar (Atari, Go — M11.6, M11.11), robotik, envanter ve dinamik fiyatlandırma (M11.2), öneri sistemleri ve reklam (çevrimiçi karar: haydutlar, M11.7–11.8), algoritmik alım satım (durağan olmayan, gürültülü ödül — dikkatli değerlendirme gerektirir), büyük dil modellerinin insan tercihleriyle hizalanması (RLHF, M4.16). Bu modülde temel taş: geleceğin yalnız şimdiye bağlı olduğu rastgele süreçler — Markov zincirleri.
:::

## 1 · Elle

<!-- çözüm -->

$\mathbf p_1=[0.9,0.1]$, $\mathbf p_2=T\mathbf p_1=[0.9\cdot0.9+0.5\cdot0.1,\;0.1\cdot0.9+0.5\cdot0.1]=[0.86,0.14]$. Durağanda $0.1\pi_0=0.5\pi_1$ → $\pi_0=5/6$.

AR(1): $\sigma^2=\phi^2\sigma^2+\sigma_w^2$ → $\sigma^2=1/(1-0.64)$; $C(\tau)=\phi^{|\tau|}\sigma^2$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**İki zar**

- `p7`: Toplamın 7 olma olasılığı
- `ex`: Toplamın beklenen değeri

**Zincir: $T=\begin{bmatrix}0.9&0.5\\0.1&0.5\end{bmatrix}$ (sütun $j$: $S_j$'den çıkış olasılıkları), $\mathbf p_0=[1,0]^T$**

- `p2`: $\mathbf p_2$'nin ilk bileşeni
- `pi0`: Durağan dağılımda $\pi_0$

**Süreçler**

- `ar`: AR(1) $x_t=0.8x_{t-1}+w_t$, $\sigma_w^2=1$: durağan varyans
- `ac`: Aynı sürecin 2 gecikmeli otokovaryansı
- `w`: Standart Wiener süreci: $W(4)$'ün standart sapması

In [ ]:
cevaplar = dict(
    p7=None,
    ex=None,
    p2=None,
    pi0=None,
    ar=None,
    ac=None,
    w=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"p7":[0.166667,0.0001],"ex":[7,0],"p2":[0.86,0],"pi0":[0.833333,0.0001],"ar":[2.777778,0.0001],"ac":[1.777778,0.0001],"w":[2,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, `random.Random(seed)` ile.

- `sample_index(probs, u)` — $u\in[0,1)$ için kümülatif olasılıklarda ilk $c_i>u$ olan $i$ (ters CDF örnekleme).
- `simulate_chain(T, s0, steps, seed=0)` — `T` liste listesi (sütun-stokastik, `T[i][j]` = $j\to i$); her adımda `rng.random()` ile sütundan örnekle. `steps+1` uzunlukta durum listesi.
- `ar1(phi, sigma, n, seed=0)` — $x_0=0$, $x_t=\phi x_{t-1}+w_t$, $w_t$ = `rng.gauss(0, sigma)`.
- `autocov(x, lag)` — örnek otokovaryansı $\frac1n\sum_{t}(x_t-\bar x)(x_{t+lag}-\bar x)$ (payda $n$).

In [ ]:
%%writefile stage_b.py
import random


def sample_index(probs, u):
    ...


def simulate_chain(T, s0, steps, seed=0):
    ...


def ar1(phi, sigma, n, seed=0):
    ...


def autocov(x, lag):
    ...


T = [[0.9, 0.5], [0.1, 0.5]]
s = simulate_chain(T, 0, 10000)
print(s.count(0) / len(s))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**.

- `n_step(T, p0, n)` — `np.linalg.matrix_power`.
- `stationary(T)` — `np.linalg.eig`, özdeğeri 1'e en yakın özvektör, gerçel kısım, toplam 1.
- `tv_curve(T, p0, n)` — $t=0..n$ için $\frac12\lVert\mathbf p_t-\boldsymbol\pi\rVert_1$ (toplam varyasyon uzaklığı); kuvvetleri tek seferde: `np.linalg.matrix_power` yerine özdeğer ayrışımı $T=V\Lambda V^{-1}$ ile $\mathbf p_t=V\Lambda^tV^{-1}\mathbf p_0$.
- `ar1_paths(phi, sigma, n, paths, seed=0)` — `(paths, n)`; `scipy.signal.lfilter([1], [1, -phi], w, axis=1)` ile döngüsüz.
- `ensemble_stats(X)` — her $t$ için topluluk ortalaması ve varyansı `(mean (n,), var (n,))`.

In [ ]:
%%writefile stage_c.py
import numpy as np
from scipy import signal


def n_step(T, p0, n):
    ...


def stationary(T):
    ...


def tv_curve(T, p0, n):
    ...


def ar1_paths(phi, sigma, n, paths, seed=0):
    ...


def ensemble_stats(X):
    ...


T = np.array([[0.9, 0.5], [0.1, 0.5]])
print(n_step(T, np.array([1.0, 0]), 2), stationary(T))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Binlerce zincir ve geçiş matrisi kestirimi (PyTorch).

- `simulate(T, s0, steps, n, seed=0)` — `T (M, M)` sütun-stokastik tensör; `n` zinciri aynı anda: her adımda `torch.multinomial(T[:, s].T, 1)` (zincir başına kendi sütunu). `(n, steps+1)` `long`.
- `count_mle(traj, M)` — geçiş sayıları $N_{ij}$ ($j\to i$), sütun normalize: $\hat T_{ij}=N_{ij}/\sum_iN_{ij}$. Döngüsüz (`index_put_` ya da `bincount`).
- `grad_mle(traj, M, steps=500, lr=0.5)` — $\hat T=\mathrm{softmax}(\Theta, \text{dim}=0)$, log-olabilirlik $\sum\log\hat T_{s_{t+1},s_t}$'yi Adam ile en çoklaştır; sayma kestirimine yakınsamalı (kapalı form ile gradyan yöntemi aynı optimumu bulur).

In [ ]:
%%writefile stage_d.py
import torch


def simulate(T, s0, steps, n, seed=0):
    ...


def count_mle(traj, M):
    ...


def grad_mle(traj, M, steps=500, lr=0.5):
    ...


T = torch.tensor([[0.9, 0.5], [0.1, 0.5]], dtype=torch.float64)
tr = simulate(T, 0, 50, 100)
print(count_mle(tr, 2))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Wiener süreci, vektörel.

- `wiener(n_paths, n_steps, t_max=1.0, seed=0)` — `(n_paths, n_steps+1)` `float64`, $W_0=0$, artışlar $\mathcal N(0,\Delta t)$, `cumsum`.
- `quadratic_variation(W)` — her yol için $\sum(\Delta W)^2$ (→ $t_{max}$).
- `max_exceed(W, a)` — $P(\max_{s\le t}W_s\ge a)$ kestirimi (`cummax` ya da `max`) ve $2P(W_t\ge a)$ kestirimi: yansıma ilkesi ikisini eşitler (ayrık adımda max hafif küçük kalır).

In [ ]:
%%writefile stage_e.py
import torch


def wiener(n_paths, n_steps, t_max=1.0, seed=0):
    ...


def quadratic_variation(W):
    ...


def max_exceed(W, a):
    ...


W = wiener(5, 100)
print(W.shape)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Soğurucu zincirler: kumarbazın iflası ve ortalama ilk geçiş zamanları

Satır-stokastik $P$ (burada $P_{ij}=P(j|i)$) ile:

- `absorption(P, absorbing)` — geçici durumlar $Q$, geçiciden soğurucuya $R$; temel matris $N=(I-Q)^{-1}$, soğurulma olasılıkları $B=NR$, beklenen adım $\mathbf t=N\mathbf 1$. `(B, t)`.
- `gamblers_ruin(N, p)` — $0..N$ servet zinciri ($p$ kazanma olasılığı), soğurucular 0 ve $N$: her başlangıç için $N$'e ulaşma olasılığı (`absorption` ile). Kapalı form: $\frac{1-(q/p)^i}{1-(q/p)^N}$ ($p\ne q$).
- `mean_first_passage(P)` — $m_{ij}$: $i$'den $j$'ye beklenen adım ($m_{jj}$ = dönüş zamanı); her hedef $j$ için $(I-P_{-j,-j})\mathbf m=\mathbf 1$ çöz (hedef döngüsü serbest), $m_{jj}=1+\sum_kP_{jk}m_{kj}$. Kac: $m_{jj}=1/\pi_j$.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def absorption(P, absorbing):
    ...


def gamblers_ruin(N, p):
    ...


def mean_first_passage(P):
    ...


print(gamblers_ruin(4, 0.5))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Durağanlık ve ergodiklik: topluluk ve zaman ortalaması

- `random_phase(n_paths, n, omega, seed=0)` — $x_t=\cos(\omega t+\theta)$, $\theta\sim U[0,2\pi)$: WSS (ortalama 0, kovaryans $\frac12\cos\omega\tau$).
- `random_amplitude(n_paths, n, omega, seed=0)` — $x_t=A\cos(\omega t)$, $A\sim\mathcal N(0,1)$: ortalama 0 ama varyans $\cos^2\omega t$ zamana bağlı → WSS **değil**.
- `random_constant(n_paths, n, seed=0)` — $x_t=A$: katı durağan ama **ergodik değil** (zaman ortalaması $A$, topluluk ortalaması 0).
- `time_vs_ensemble(X)` → `(zaman ortalamalarının std'si, topluluk ortalamalarının std'si)`.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def random_phase(n_paths, n, omega, seed=0):
    ...


def random_amplitude(n_paths, n, omega, seed=0):
    ...


def random_constant(n_paths, n, seed=0):
    ...


def time_vs_ensemble(X):
    ...


print(random_phase(3, 5, 0.3).shape)

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Karakter düzeyinde Markov dil modeli (k. derece)

Markov dil modeli: bir sonraki karakter yalnız son $k$ karaktere bağlı.

- `fit(text, k)` — bağlam (k karakterlik dize) → sonraki karakter sayıları sözlüğü.
- `log_prob(model, text, k, alpha=1.0, vocab=None)` — Laplace yumuşatmalı ($+\alpha$) toplam log-olasılık (doğal log), ilk $k$ karakter hariç.
- `generate(model, seed_text, n, k, seed=0)` — sayılara orantılı örnekleme; bağlam görülmemişse sözlükteki rastgele bir bağlamla devam et.

In [ ]:
%%writefile extra_x3.py
import math
import random
from collections import Counter, defaultdict


def fit(text, k):
    ...


def log_prob(model, text, k, alpha=1.0, vocab=None):
    ...


def generate(model, seed_text, n, k, seed=0):
    ...


m = fit("abracadabra", 1)
print(m["a"])

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: PageRank: web grafiğinde durağan dağılım

[Page, Brin, Motwani, Winograd — The PageRank Citation Ranking: Bringing Order to the Web (Stanford, 1999)](http://ilpubs.stanford.edu:8090/422/)

Rastgele sörfçü modelini bir Markov zinciri olarak kur (sönümleme faktörü d = 0.85, sarkan düğümler için düzgün atlama), PageRank'i kuvvet yinelemesiyle hesapla; seyrek matrislerle (SciPy CSR) milyon kenarlı bir grafikte (ör. SNAP web-Google) çalıştır ve yakınsama hızının d'ye bağımlılığını ikinci özdeğerle ilişkilendir.


**Adımlar**

1. Seyrek, sütun-stokastik geçiş matrisi; sarkan düğüm düzeltmesi.
2. Kuvvet yinelemesi; L1 değişim < 1e-10'a kadar iterasyon sayısı.
3. d = 0.5, 0.85, 0.99 için iterasyon sayısı ve $|\lambda_2|\le d$ ilişkisi.
4. Kişiselleştirilmiş PageRank (atlama dağılımı bir düğüm kümesine yoğun).

**Kabul ölçütleri**

- SNAP grafiğinde ilk 20 düğüm ve NetworkX `pagerank` ile 1e-6 içinde uyum.
- d'ye karşı iterasyon grafiği ve teorik $\log\varepsilon/\log d$ eğrisi.

## Kaynaklar

- [Sutton & Barto — Reinforcement Learning: An Introduction (2. baskı), Bölüm 3](http://incompleteideas.net/book/the-book-2nd.html) — CC BY-NC-ND 2.0
- [Papoulis & Pillai — Probability, Random Variables and Stochastic Processes (4. baskı)](https://www.mheducation.com/) — ©
- [Grinstead & Snell — Introduction to Probability, Bölüm 11 (Markov zincirleri)](https://math.dartmouth.edu/~prob/prob/prob.pdf) — GFDL
- [Levin & Peres — Markov Chains and Mixing Times (2. baskı)](https://pages.uoregon.edu/dlevin/MARKOV/) — ücretsiz PDF